# Day 10 — Solution: Permutation Tests

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from scipy import stats
DATA_SOURCE = os.environ.get("QRC_DATA", "real")
from qrc.data import get_prices
from qrc.synth import synthetic_prices

if DATA_SOURCE == "real":
    px = get_prices(["SPY", "TLT"], start="1990-01-01")
else:
    px = synthetic_prices(n_days=4000, n_assets=2, seed=61, corr=0.15)
    px.columns = ["SPY", "TLT"]
    px["TLT"] = -px["TLT"]      # model the negative link on the synth side
r = px.pct_change().dropna()
rng = np.random.default_rng(31)

## E1 — the engine, validated

In [ ]:
def perm_test_two_sample(x, y, B=1000, rng=rng):
    x, y = np.asarray(x, float), np.asarray(y, float)
    pooled = np.concatenate([x, y])
    n1 = len(x)
    obs = x.mean() - y.mean()
    count = 0
    for _ in range(B):
        perm = rng.permutation(pooled)
        if abs(perm[:n1].mean() - perm[n1:].mean()) >= abs(obs):
            count += 1
    return (count + 1) / (B + 1)

# (a) same data: permutation p vs Welch p
a = r.iloc[:, 0].values[:1500]; b = r.iloc[:, 1].values[:1500]
p_perm = perm_test_two_sample(a, b, B=1500)
p_welch = stats.ttest_ind(a, b, equal_var=False).pvalue
print(f"p permutation {p_perm:.3f} vs Welch {p_welch:.3f}")

# (b) null p-values look uniform: 200 pairs, B=500
rng2 = np.random.default_rng(7)
ps = [perm_test_two_sample(rng2.normal(size=200), rng2.normal(size=200), B=500, rng=rng2)
      for _ in range(200)]
ps = np.array(ps)
print(f"mean p {ps.mean():.3f} (want 0.5); frac < 0.05: {(ps < 0.05).mean():.2f} (want ~0.05)")

# (c) power: shift by 0.4 SD
sig = rng2.normal(0, 0.01, 200); null = rng2.normal(0, 0.01, 200)
rng3 = np.random.default_rng(8)
rej = np.mean([perm_test_two_sample(sig, null + 0.4*0.01, B=500, rng=rng3) < 0.05
               for _ in range(100)])
print(f"rejection rate at true shift 0.4 SD: {rej:.2f} (null rate ~0.05)")

**Expected reasoning.** (a) The two p's agree to Monte-Carlo precision
(both are testing the same null on the same data; permutation is the
more robust one — it needs only exchangeability, not normality).
(b) Mean p ≈ 0.5, rejection ≈ 5%: under the null the p is *uniform*,
so "p < 0.05" happens 5% of the time — that is the multiple-testing
fact of module 04.11 with your own eyes. (c) 0.4 SD at n = 200 is
detectable (rejection typically 0.35–0.6): permutation tests inherit
the same power constraints as t-tests — no free lunch, just fewer
assumptions.

## E2 — correlation by permutation

In [ ]:
x = r.iloc[:, 0].values; y = r.iloc[:, 1].values   # already date-aligned
rho_obs = np.corrcoef(x, y)[0, 1]
B = 2000
rhos = np.array([np.corrcoef(x, rng.permutation(y))[0, 1] for _ in range(B)])
p = (np.sum(np.abs(rhos) >= abs(rho_obs)) + 1) / (B + 1)
t_fisher = rho_obs * np.sqrt((len(x) - 2) / (1 - rho_obs**2))
p_fisher = 2 * stats.t.sf(abs(t_fisher), len(x) - 2)
print(f"rho {rho_obs:.3f} | permutation p {p:.3f} | Fisher-z p {p_fisher:.3f}")

**Expected reasoning.** For real SPY/TLT: rho ≈ −0.2 to −0.35, both p's
tiny — the negative asset-class link is as solid as asset-class facts
get. Permuting y against fixed x tests the sharp null "the pairing of
dates carries no linear information" — approximately valid here
because *returns* are near-iid (their ACF ≈ 0); had we done this on
*prices* (strongly autocorrelated), the shuffled null would be wrong
and the p uninterpretable. Part (c)'s point: permutation tests carry
the data's dependence assumptions quietly — check ACF first, always.

## E3 — the dependence test, done right

In [ ]:
a = np.abs(r.iloc[:, 0]).values
rho1 = pd.Series(a).autocorr(1)

def acf_stat(v):
    v = v - v.mean()
    return np.corrcoef(v[:-1], v[1:])[0, 1]

B = 2000
p_naive = np.mean([abs(acf_stat(rng.permutation(a))) >= abs(rho1) for _ in range(B)])
b = 22
def block_shuffle(v, b=b, rng=rng):
    n = len(v)
    nb = int(np.ceil(n / b))
    starts = rng.integers(0, n, size=nb)
    return np.concatenate([v[s:s+b] for s in starts])[:n]
p_block = np.mean([abs(acf_stat(block_shuffle(a))) >= abs(rho1) for _ in range(B)])
print(f"rho1(|r|) = {rho1:.3f} | p naive-shuffle {p_naive:.4f} | p block(22) {p_block:.4f}")

**Expected reasoning.** rho1(|r|) ≈ 0.15–0.25. The naive p is often 0
or near-0 (B+1 coin flips); the block p is larger — it allows the
null world to have clustering. The gap between them is *exactly* the
contribution of the clustering you destroyed: how much more extreme
the statistic looks when the null is too quiet. If p_block is still
small (it usually is, on both real and synthetic clustered data), you
have shown something real: the clustering is not an artifact of the
test — it is a *stylized fact*, present by construction in the
synthetic generator and in the actual market. The lesson lands in both
directions: naive shuffling manufactures significance, and honest
shuffling can still find the real thing.

## E4 — design a placebo (exemplar)

**Null to build:** the characteristic carries no information → which
stock lands in the top decile is arbitrary *within each rebalance date*.
Permute the labels: at each rebalance, assign the "top decile" label to
a random decile of stocks (same date, same cross-section), recompute
the decile's forward returns over all rebalances, B = 500, and ask
whether the observed 4% annual edge falls in the null distribution.
**What a rejection establishes:** the *labeling rule* picks
above-average returns — the characteristic sorts returns, beyond the
cross-sectional null. **What it does NOT establish:** that the edge
survives costs, that it is not one of 100 correlated characteristics
tested (module 04.11), that it is not a known factor in disguise
(module 07), or that it will continue — the permutation null is a
statement about *this sample's* labeling, not about the future.